[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yryo1005/Deep_Variation_Information_Bottleneck_training/blob/master/ex002_mnist_ae.ipynb)

> **実行する前に，ランタイムを GPU に変更すること．**  
> Google Colab ではメニューの **ランタイム → ランタイムのタイプを変更** を開き，ハードウェア アクセラレータを **GPU** に設定して保存する．CPU のままだと学習に時間がかかる．

# 実験002: Autoencoder による MNIST の再構成

本ノートブックでは，PyTorch を用いて MNIST 手書き数字を再構成する Autoencoder を学習する．

この講座では，次の順でプログラムを積み上げる．

1. NN による MNIST 分類（交差エントロピー誤差）
2. **本実験**: Autoencoder による MNIST の再構成
3. Variational Autoencoder による MNIST の再構成
4. Deep Variational Information Bottleneck による MNIST 分類
5. 2 次元特徴マップの可視化
6. 敵対的学習（ノイズ最適化）
7. Deep VIB + Brier スコアによる MNIST 分類
8. Deep VIB + クラス間重み付き CCE による MNIST 分類

学習ループの分割は実験001と同じである．本実験での変更点は，教師信号をクラスラベルから入力画像自身へ変え，モデルを Autoencoder にすることである．


## 1. ライブラリ


In [ ]:
import json
import os
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from tqdm.auto import tqdm


## 2. 再現性とログ


In [ ]:
def set_seed(seed=0):
    """実験の再現性を担保するため，乱数シードを固定する．

    Args:
        seed (int): 固定するシード値．デフォルトは 0．

    Returns:
        None
    """
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    torch.use_deterministic_algorithms(True)


class ResultLogger:
    """学習中の指標（Loss，MAE など）を記録し，JSON として保存・読み込みするクラス．

    Args:
        target_path (str | None): 既存ログ JSON のパス．指定時は初期化時に読み込む．

    Returns:
        なし（インスタンスを生成する）
    """

    def __init__(self, target_path=None):
        self.names = None
        self.history = {}
        if target_path:
            self.load(target_path)

    def set_names(self, *names):
        """記録する指標名を登録する．

        Args:
            *names (str): 指標名．例: "train_loss", "train_mae"．

        Returns:
            None
        """
        if self.names:
            raise RuntimeError("指標名はすでに登録されている．")
        self.names = list(names)
        for name in self.names:
            if name not in self.history:
                self.history[name] = []

    def __call__(self, *values):
        """1 エポック分の指標値を履歴へ追加する．

        Args:
            *values (float | int): set_names で登録した順の値．

        Returns:
            None
        """
        if self.names is None:
            raise RuntimeError("先に set_names で指標名を登録すること．")
        if len(values) != len(self.names):
            raise RuntimeError("値の数が登録済み指標名の数と一致しません．")
        for name, value in zip(self.names, values):
            self.history[name].append(value)

    def save(self, target_path):
        """履歴を JSON ファイルとして保存する．

        Args:
            target_path (str): 保存先パス．

        Returns:
            None
        """
        with open(target_path, "w") as f:
            json.dump(self.history, f, indent=4)

    def load(self, target_path):
        """JSON ファイルから履歴を読み込む．

        Args:
            target_path (str): 読み込む JSON のパス．

        Returns:
            None
        """
        with open(target_path, "r") as f:
            data = json.load(f)
        self.history = data
        self.names = list(data.keys())

    def __getitem__(self, key):
        """指定した指標の履歴リストを取得する．

        Args:
            key (str): 指標名．

        Returns:
            history (list): 指標の履歴．存在しない場合は空リスト．
        """
        return self.history.get(key, [])


## 3. 実験設定

`outputs/ex002_mnist_ae/{最適化手法}/{学習率}_{バッチサイズ}/{seed}/`

本実験はサンプルのため，シードは 1 つ（`seed=0`）とする．


In [ ]:
EXPERIMENT_NAME = "ex002_mnist_ae"

NOTEBOOK_DIR = Path.cwd().resolve()
if NOTEBOOK_DIR.name == EXPERIMENT_NAME:
    PROJECT_ROOT = NOTEBOOK_DIR.parent
else:
    PROJECT_ROOT = NOTEBOOK_DIR

DATASET_DIR = PROJECT_ROOT / "datasets" / EXPERIMENT_NAME / "standard"
OUTPUT_ROOT = PROJECT_ROOT / "outputs" / EXPERIMENT_NAME

EPOCHS = 10
SEEDS = [0]
OPTIMIZER_NAMES = ["Adam"]
LEARNING_RATES = [0.001]
BATCH_SIZES = [128]

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"DATASET_DIR : {DATASET_DIR}")
print(f"OUTPUT_ROOT : {OUTPUT_ROOT}")


## 4. モデルの定義

Autoencoder は Encoder と Decoder の 2 つのネットワークからなる．

- Encoder: 入力画像を低次元の潜在ベクトル $z$ へ圧縮する（784 -> 256 -> 128 -> 32）
- Decoder: $z$ から元の画像サイズへ戻す（32 -> 128 -> 256 -> 784）

出力には Sigmoid を用い，画素値を $[0, 1]$ に収める．潜在次元は 32 とする．2 次元での可視化は後の実験で扱う．


In [ ]:
class MNISTAutoencoder(nn.Module):
    """MNIST を再構成する全結合 Autoencoder．

    Encoder は 784 -> 256 -> 128 -> 32，Decoder は 32 -> 128 -> 256 -> 784 の線形変換を行う．
    再構成画像の各画素は Sigmoid により [0, 1] に収める．

    Args:
        なし（層の次元はクラス内で固定する）
    """

    def __init__(self):
        super().__init__()
        self.enc1 = nn.Linear(784, 256)
        self.enc2 = nn.Linear(256, 128)
        self.enc3 = nn.Linear(128, 32)
        self.dec1 = nn.Linear(32, 128)
        self.dec2 = nn.Linear(128, 256)
        self.dec3 = nn.Linear(256, 784)

    def encode(self, x):
        """入力画像を潜在ベクトルへ圧縮する．

        Args:
            x (torch.Tensor): 入力画像．形状は (N, 1, 28, 28) または (N, 784)．

        Returns:
            z (torch.Tensor): 潜在ベクトル．形状は (N, 32)．
        """
        x = x.view(x.size(0), -1)
        x = F.relu(self.enc1(x))
        x = F.relu(self.enc2(x))
        z = self.enc3(x)
        return z

    def decode(self, z):
        """潜在ベクトルから画像を再構成する．

        Args:
            z (torch.Tensor): 潜在ベクトル．形状は (N, 32)．

        Returns:
            recon (torch.Tensor): 再構成画像．形状は (N, 1, 28, 28)．
        """
        x = F.relu(self.dec1(z))
        x = F.relu(self.dec2(x))
        x = torch.sigmoid(self.dec3(x))
        recon = x.view(-1, 1, 28, 28)
        return recon

    def forward(self, x):
        """入力画像を再構成する．

        Args:
            x (torch.Tensor): 入力画像．形状は (N, 1, 28, 28) または (N, 784)．

        Returns:
            recon (torch.Tensor): 再構成画像．形状は (N, 1, 28, 28)．
        """
        z = self.encode(x)
        recon = self.decode(z)
        return recon


def load_model(ModelClass, weight_path=None, seed=0):
    """モデルをインスタンス化し，必要なら学習済み重みを読み込む．

    Args:
        ModelClass (type): torch.nn.Module を継承したモデルクラス．
        weight_path (str | None): 学習済み重み (.pth) のパス．None なら初期値を使う．
        seed (int): パラメータ初期化用の乱数シード．

    Returns:
        model (torch.nn.Module): インスタンス化されたモデル．
    """
    set_seed(seed)
    model = ModelClass()
    if weight_path is not None:
        state_dict = torch.load(weight_path, map_location="cpu")
        model.load_state_dict(state_dict)
    return model


## 5. データセットと DataLoader

MNIST は公式の学習 60,000 枚 / テスト 10,000 枚に分割されている．画素値は `ToTensor()` により $[0, 1]$ に正規化する．DataLoader のシャッフルは `torch.Generator` で決定論的にする．


In [ ]:
def load_dataloader(seed=0, batch_size=128):
    """MNIST の学習用・検証用 DataLoader を作成する．

    Args:
        seed (int): データの並びを固定する乱数シード．
        batch_size (int): ミニバッチサイズ．

    Returns:
        train_dataloader (torch.utils.data.DataLoader): 学習用 DataLoader．
            各バッチは画像 (N, 1, 28, 28) とラベル (N,) のタプル．
        test_dataloader (torch.utils.data.DataLoader): 検証用 DataLoader．
            各バッチの形状は学習用と同じ．
    """
    set_seed(seed)
    DATASET_DIR.mkdir(parents=True, exist_ok=True)

    transform = transforms.ToTensor()
    train_dataset = datasets.MNIST(
        root=str(DATASET_DIR),
        train=True,
        download=True,
        transform=transform,
    )
    test_dataset = datasets.MNIST(
        root=str(DATASET_DIR),
        train=False,
        download=True,
        transform=transform,
    )

    generator = torch.Generator()
    generator.manual_seed(seed)

    train_dataloader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        generator=generator,
        num_workers=0,
    )
    test_dataloader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
    )
    return train_dataloader, test_dataloader


### データの確認

学習用 DataLoader から 1 バッチを取り出し，画像の形状とラベルの例を表示する．Autoencoder の学習ではラベルは使わず，画像自身を教師信号とする．


In [ ]:
preview_loader, _ = load_dataloader(seed=0, batch_size=8)
preview_images, preview_labels = next(iter(preview_loader))
print(f"images shape: {tuple(preview_images.shape)}")
print(f"labels shape: {tuple(preview_labels.shape)}")
print(f"labels      : {preview_labels.tolist()}")

fig, axes = plt.subplots(1, 8, figsize=(16, 2.5))
for i, ax in enumerate(axes):
    ax.imshow(preview_images[i, 0], cmap="gray")
    ax.set_title(f"label={preview_labels[i].item()}")
    ax.axis("off")
plt.tight_layout()
plt.show()


## 6. 誤差関数と評価関数

再構成タスクでは平均二乗誤差（MSE）を用いる．教師信号は入力画像自身である．

評価指標は平均絶対誤差（MAE）とする．Loss の計算は `iteration` 内で行うため，`metrics_func` では MAE のみを返す．


In [ ]:
def loss_func(outputs, teacher_signals):
    """再構成画像と入力画像の平均二乗誤差を計算する．

    Args:
        outputs (torch.Tensor): 再構成画像．形状は (N, 1, 28, 28)．
        teacher_signals (torch.Tensor): 入力画像（教師信号）．形状は (N, 1, 28, 28)．

    Returns:
        loss (torch.Tensor): 1 画素あたりの平均二乗誤差．形状は ()．
    """
    loss = F.mse_loss(outputs, teacher_signals, reduction="mean")
    return loss


def metrics_func(outputs, teacher_signals):
    """再構成画像と入力画像から MAE を計算する．

    Args:
        outputs (torch.Tensor): 再構成画像．形状は (N, 1, 28, 28)．
        teacher_signals (torch.Tensor): 入力画像（教師信号）．形状は (N, 1, 28, 28)．

    Returns:
        metrics_to_value (dict): キー "mae" に 1 画素あたりの平均絶対誤差 (float) を格納した辞書．
    """
    mae = (outputs - teacher_signals).abs().mean().item()
    return {"mae": mae}


## 7. 学習ループ

学習は次の 3 段に分ける．ネストを浅く保つため，役割を関数ごとに固定する．

1. `iteration`: 1 ミニバッチの順伝播・誤差計算・（学習時のみ）パラメータ更新
2. `epoch`: DataLoader 全件を処理し，1 データあたりの平均指標を返す
3. `train`: エポックを繰り返し，最良モデルとログを保存する

実験001との違いは，`epoch` 内でラベルを捨て，**入力画像を教師信号として渡す**点である．


In [ ]:
def iteration(model, inputs, teacher_signals, optimizer=None):
    """1 ミニバッチを学習または検証する．

    Args:
        model (torch.nn.Module): 学習 / 検証対象のモデル．
        inputs (torch.Tensor): 入力画像．形状は (N, 1, 28, 28)．
        teacher_signals (torch.Tensor): 再構成の教師信号（入力画像自身）．形状は (N, 1, 28, 28)．
        optimizer (torch.optim.Optimizer | None): 最適化手法．None なら検証モード．

    Returns:
        metrics_to_value (dict): "loss" と "mae" をキーとする 1 データあたりの平均値の辞書．
    """
    if optimizer is None:
        with torch.no_grad():
            outputs = model(inputs)
            loss = loss_func(outputs, teacher_signals)
    else:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = loss_func(outputs, teacher_signals)
        loss.backward()
        optimizer.step()

    metrics_to_value = metrics_func(outputs, teacher_signals)
    metrics_to_value["loss"] = loss.item()
    return metrics_to_value


def epoch(model, dataloader, optimizer=None):
    """DataLoader 全件を 1 周し，1 データあたりの平均指標を返す．

    Args:
        model (torch.nn.Module): 学習 / 検証対象のモデル．
        dataloader (torch.utils.data.DataLoader): 入力とラベルの DataLoader．
        optimizer (torch.optim.Optimizer | None): 最適化手法．None なら検証モード．

    Returns:
        metrics_to_value (dict): エポック全体の 1 データあたり平均（"loss", "mae"）．
    """
    if optimizer is None:
        model.eval()
    else:
        model.train()

    device = next(model.parameters()).device
    sum_metrics = {}
    n_total = 0

    progress = tqdm(dataloader, leave=False)
    for inputs, labels in progress:
        inputs = inputs.to(device)
        teacher_signals = inputs

        batch_metrics = iteration(model, inputs, teacher_signals, optimizer)
        batch_size = inputs.size(0)
        n_total += batch_size

        for name, value in batch_metrics.items():
            sum_metrics[name] = sum_metrics.get(name, 0.0) + value * batch_size

        average_metrics = {name: total / n_total for name, total in sum_metrics.items()}
        progress.set_postfix({name: f"{value:.4f}" for name, value in average_metrics.items()})

    metrics_to_value = {name: total / n_total for name, total in sum_metrics.items()}
    return metrics_to_value


In [ ]:
def build_optimizer(model, optimizer_name, lr):
    """最適化手法の名前から Optimizer を生成する．

    Args:
        model (torch.nn.Module): パラメータを更新するモデル．
        optimizer_name (str): 最適化手法名．"Adam" または "SGD"．
        lr (float): 学習率．

    Returns:
        optimizer (torch.optim.Optimizer): 生成された Optimizer．
    """
    if optimizer_name == "Adam":
        return torch.optim.Adam(model.parameters(), lr=lr)
    if optimizer_name == "SGD":
        return torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    raise ValueError(f"未対応の最適化手法である: {optimizer_name}")


def train(
    target_dir,
    ModelClass,
    load_dataloader,
    epochs,
    batch_size,
    seed=0,
    lr=0.001,
    optimizer_name="Adam",
):
    """モデルを学習し，最良重みとログを保存する．

    Args:
        target_dir (str): 結果の保存先ディレクトリ．
        ModelClass (type): 学習するモデルクラス．
        load_dataloader (callable): seed と batch_size から DataLoader を返す関数．
        epochs (int): 学習エポック数．
        batch_size (int): ミニバッチサイズ．
        seed (int): 乱数シード．
        lr (float): 学習率．
        optimizer_name (str): 最適化手法名．

    Returns:
        None
    """
    os.makedirs(target_dir, exist_ok=True)
    train_dataloader, test_dataloader = load_dataloader(seed=seed, batch_size=batch_size)
    model = load_model(ModelClass, weight_path=None, seed=seed)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    optimizer = build_optimizer(model, optimizer_name, lr)

    logger = ResultLogger()
    logger.set_names("train_loss", "train_mae", "val_loss", "val_mae")

    train_metrics = epoch(model, train_dataloader, optimizer=None)
    val_metrics = epoch(model, test_dataloader, optimizer=None)
    logger(train_metrics["loss"], train_metrics["mae"], val_metrics["loss"], val_metrics["mae"])
    print(
        f"[epoch 0/{epochs}] "
        f"train_loss={train_metrics['loss']:.4f} train_mae={train_metrics['mae']:.4f} "
        f"val_loss={val_metrics['loss']:.4f} val_mae={val_metrics['mae']:.4f}"
    )

    best_val_loss = val_metrics["loss"]
    torch.save(model.state_dict(), os.path.join(target_dir, "best_model.pth"))

    for epoch_idx in range(1, epochs + 1):
        train_metrics = epoch(model, train_dataloader, optimizer=optimizer)
        val_metrics = epoch(model, test_dataloader, optimizer=None)
        logger(train_metrics["loss"], train_metrics["mae"], val_metrics["loss"], val_metrics["mae"])
        print(
            f"[epoch {epoch_idx}/{epochs}] "
            f"train_loss={train_metrics['loss']:.4f} train_mae={train_metrics['mae']:.4f} "
            f"val_loss={val_metrics['loss']:.4f} val_mae={val_metrics['mae']:.4f}"
        )

        if val_metrics["loss"] < best_val_loss:
            best_val_loss = val_metrics["loss"]
            torch.save(model.state_dict(), os.path.join(target_dir, "best_model.pth"))

        logger.save(os.path.join(target_dir, "log.json"))

    logger.save(os.path.join(target_dir, "log.json"))
    print(f"best val_loss={best_val_loss:.4f}")
    print(f"saved: {target_dir}")


## 8. 実行


In [ ]:
print(f"device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

for optimizer_name in OPTIMIZER_NAMES:
    for lr in LEARNING_RATES:
        for batch_size in BATCH_SIZES:
            for seed in SEEDS:
                hyperparam_name = f"{lr}_{batch_size}"
                target_dir = OUTPUT_ROOT / optimizer_name / hyperparam_name / str(seed)
                log_path = target_dir / "log.json"
                weight_path = target_dir / "best_model.pth"

                if log_path.exists() and weight_path.exists():
                    print(f"skip: {target_dir}")
                    continue

                print(f"train: {target_dir}")
                train(
                    target_dir=str(target_dir),
                    ModelClass=MNISTAutoencoder,
                    load_dataloader=load_dataloader,
                    epochs=EPOCHS,
                    batch_size=batch_size,
                    seed=seed,
                    lr=lr,
                    optimizer_name=optimizer_name,
                )


## 9. 学習曲線の確認

保存したログから Loss と MAE の推移を描画する．複数シードの平均と標準偏差は，本ノートブック末尾のセルで描画する．


In [ ]:
log_path = OUTPUT_ROOT / OPTIMIZER_NAMES[0] / f"{LEARNING_RATES[0]}_{BATCH_SIZES[0]}" / str(SEEDS[0]) / "log.json"
logger = ResultLogger(str(log_path))
epochs_axis = list(range(len(logger["train_loss"])))

fig, axes = plt.subplots(1, 2, figsize=(10, 5))

axes[0].plot(epochs_axis, logger["train_loss"], label="train")
axes[0].plot(epochs_axis, logger["val_loss"], label="val")
axes[0].set_xlabel("epoch")
axes[0].set_ylabel("loss")
axes[0].set_title("Reconstruction MSE")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_axis, logger["train_mae"], label="train")
axes[1].plot(epochs_axis, logger["val_mae"], label="val")
axes[1].set_xlabel("epoch")
axes[1].set_ylabel("mae")
axes[1].set_title("Reconstruction MAE")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"final train_loss = {logger['train_loss'][-1]:.4f}")
print(f"final val_loss   = {logger['val_loss'][-1]:.4f}")
print(f"best val_loss    = {min(logger['val_loss']):.4f}")
print(f"final train_mae  = {logger['train_mae'][-1]:.4f}")
print(f"final val_mae    = {logger['val_mae'][-1]:.4f}")
print(f"best val_mae     = {min(logger['val_mae']):.4f}")


## 10. 再構成画像の確認

最良モデルで検証画像を再構成し，上段に入力，下段に再構成を並べて表示する．数字の形が残っていれば，潜在ベクトル $z$ に画像の情報が圧縮されている．


In [ ]:
weight_path = OUTPUT_ROOT / OPTIMIZER_NAMES[0] / f"{LEARNING_RATES[0]}_{BATCH_SIZES[0]}" / str(SEEDS[0]) / "best_model.pth"
_, test_loader = load_dataloader(seed=SEEDS[0], batch_size=8)
images, labels = next(iter(test_loader))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = load_model(MNISTAutoencoder, weight_path=str(weight_path), seed=SEEDS[0])
model = model.to(device)
model.eval()
with torch.no_grad():
    reconstructions = model(images.to(device)).cpu()

fig, axes = plt.subplots(2, 8, figsize=(16, 4.5))
for i in range(8):
    axes[0, i].imshow(images[i, 0], cmap="gray")
    axes[0, i].set_title(f"input y={labels[i].item()}")
    axes[0, i].axis("off")
    axes[1, i].imshow(reconstructions[i, 0], cmap="gray")
    axes[1, i].set_title("recon")
    axes[1, i].axis("off")
plt.tight_layout()

figure_dir = OUTPUT_ROOT / "figures"
figure_dir.mkdir(parents=True, exist_ok=True)
save_path = figure_dir / "reconstructions.png"
fig.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"saved: {save_path}")
